## **Chemical Equilibrium - 3 - A Chemical Kinetics Perspective**

<div align="left">
  <table border="1" cellpadding="6" cellspacing="0">
    <tr>
      <td bgcolor="#444444">
        <font color="#ffeb3b"><tt><b>Last updated (YYYY-MM-DD): 2026-01-08</b></tt></font>
      </td>
    </tr>
  </table>
</div>



### **Preparing the Computational Environment**  

*The next cells prepare the environment by installing the necessary packages, importing libraries, and loading constants and functions. Make sure to run them all.*

⏳ **Note:** This may take **a few minutes** to complete. ⏳  

In [ ]:
#@title <small> 💻 Run to install needed libraries <small> { display-mode: "form" }
%%capture

# --- system (for LaTeX text in matplotlib) --- #
! sudo apt update -y
! sudo apt install -y cm-super dvipng texlive-latex-extra texlive-latex-recommended

# --- Python: install and update core scientific libraries --- #

# versions that are known to work well together will be selected to avoid compatibility issues
%pip install "numpy==2.0.2"
%pip install "scipy==1.16.3"


In [ ]:
#@title <small> 💻 Load Constants and Some Helper Functions <small> { display-mode: "form" }


# ====    Import libraries of interest    ==== #
import numpy                as np
import matplotlib.pyplot    as plt
import matplotlib.ticker    as ticker
import ipywidgets           as w
# -------------------------------------------- #
from IPython.display import display
# -------------------------------------------- #
from google.colab    import files
from google.colab    import output
# -------------------------------------------- #
from   scipy.optimize import root_scalar
# -------------------------------------------- #
last_fig  = None
last_info = None
# -------------------------------------------- #
R    = 8.31447 # J / K / mol
P_o  = 1E5     # 1bar    = 1E5 Pa
c_o  = 1E3     # 1 mol/L = 1E3 mol/m^3
ZERO = 1E-14
# -------------------------------------------- #
NPOINTS   = 250  # number of time points
RELDIFF   = 1.00 # 1%
REL_XI_EQ = 0.98 # equil. at REL_XI_EQ * xi_eq
# -------------------------------------------- #


#=====================================================
def _on_download_clicked(_,conditions):
    # --- get figure from global variable ---
    if last_fig is None:
        print("No figure yet; move a slider to generate the plot...")
        return
    init_conditions = f"{T_slider.value:.0f}K_{P_slider.value:.2f}bar_{V_slider.value:.2f}L_{yA_slider.value:.2f}"
    if   conditions == "TP": fname = rf"chemkinTP_{init_conditions:s}.svg"
    elif conditions == "TV": fname = rf"chemkinTV_{init_conditions:s}.svg"
    else                   : return

    fig = last_fig
    original_size = fig.get_size_inches()
    fig.set_size_inches(10,8)
    fig.savefig(fname, bbox_inches='tight', dpi=300)
    fig.set_size_inches(original_size)
    files.download(fname)
#=====================================================

#=====================================================
def get_constants(T):
    # Equilibrium constant
    DGo   = DHo_ref - T * DSo_ref
    DGo  += DCPo_ref  * ( T - T_ref + T*np.log(T_ref / T))
    Kp_o  = np.exp(-DGo/R/T)
    Kc_o  = Kp_o * P_o/(c_o*R*T)     # adimensional
    Kc    = Kc_o * c_o # mol/m^3
    # Forward rate constant
    kfw   = arrhenius_A * np.exp(-arrhenius_B/T)
    # Backward rate constant
    kbw  = kfw/Kc
    # # REMOVE: use data from reference for now
    # if False:
    #    T,Kp_o,kfw = 298,0.144,5.251e4
    #    Kc    = Kp_o * P_o/(R*T) # mol/m^3
    #    Kc_o  = Kc/c_o           # adimensional
    #    kbw   = kfw/Kc
    # String with data
    string  = rf"   * Constants for the reaction at {T:.2f} K" + "\n"
    string += "\n"
    string += rf"     equilibrium constant   Kp^o = {Kp_o:.3E}" + "\n"
    string += rf"     equilibrium constant   Kc^o = {Kc_o:.3E}" + "\n"
    string += rf"     forward  rate constant kfw  = {kfw:.3E} 1/s" + "\n"
    string += rf"     backward rate constant kbw  = {kbw:.3E} m^3 / mol / s" + "\n"
    string += "\n"
    # return data
    return DGo,Kp_o,Kc_o,kfw,kbw,string
#-----------------------------------------------------
def limits_xi(n_0,nus):

    # maximum value of xi (calculated considering consumption of reactants)
    xi_max = min([-n_0_i/nu_i for n_0_i,nu_i in zip(n_0,nus) if nu_i < 0])

    # minimum value of xi (calculated considering consumption of products)
    xi_min = max([-n_0_i/nu_i for n_0_i,nu_i in zip(n_0,nus) if nu_i > 0])

    # Ensure numerical zeros are displayed as +0.0 for clarity
    if xi_min == -0.0: xi_min = 0.0
    if xi_max == -0.0: xi_max = 0.0

    return xi_min,xi_max
#-----------------------------------------------------
def solution_diff_eq(t,A,x0,x1,x2):
    num = x1*(x0-x2) - x2*(x0-x1) * np.exp(-A*(x1-x2)*t)
    den =    (x0-x2) -    (x0-x1) * np.exp(-A*(x1-x2)*t)
    return num/den
#-----------------------------------------------------
def string_conditions(T,p,V,xi,n,nA,yA,pA,cA,nB,yB,pB,cB,Kp_o,DG=None,DA=None):
    string  = rf"       temperature = {T:6.2f} K"+"\n"
    string += rf"       pressure    = {p*1E-5:6.2f} bar"+"\n"
    string += rf"       volume      = {V*1E3:6.2f} L"+"\n"
    string += "\n"
    string += rf"       num. moles  = {n:6.3f} mol"+"\n"
    string += rf"       extent (xi) = {xi:6.3f} mol"+"\n"
    string += "\n"
    if DG is not None:
       string += rf"       G(xi)-G(0)  = {DG/(R*T):6.3f}*(RT) mol"+"\n"
       string += "\n"
    if DA is not None:
       string += rf"       A(xi)-A(0)  = {DA/(R*T):6.3f}*(RT) mol"+"\n"
       string += "\n"
    string += rf"       data for N2O4"+"\n"
    string += rf"         - number of moles  = {nA:6.3f} mol"+"\n"
    string += rf"         - mole fraction    = {yA:6.3f} mol"+"\n"
    string += rf"         - partial pressure = {pA*1E-5:6.3f} bar"+"\n"
    string += rf"         - concentration    = {cA/1000:8.2E} M"+"\n"
    string += "\n"
    string += rf"       data for NO2"+"\n"
    string += rf"         - number of moles  = {nB:6.3f} mol"+"\n"
    string += rf"         - mole fraction    = {yB:6.3f} mol"+"\n"
    string += rf"         - partial pressure = {pB*1E-5:6.3f} bar"+"\n"
    string += rf"         - concentration    = {cB/1000:8.2E} M"+"\n"
    string += "\n"
    if pA == 0.0: string += rf"       ==> Qp^o = infinity"+"\n"
    else        : string += rf"       ==> Qp^o = {(pB*pB)/(pA*P_o):.3E}"+"\n"
    string += rf"       ==> Kp^o = {Kp_o:.3E}"+"\n"
    string += "\n"
    return string
#=====================================================

#=====================================================
def plot_data(T,data,equilibrium,xi_min,xi_max):

    plt.rcParams['text.usetex'] = True
    fig, axs = plt.subplots(2, 2, figsize=(10,8))

    # select good units for time (among secs, milisecs, microsecs and nanosecs)
    for unitst,factor in [("s",1E0) , ("ms",1E3) , ("$\\mu$s",1E6) , ("ns",1E9)]:
        last_t = data["t"][-1]*factor
        if last_t > 0.5: break
    data["t"] = [ii*factor for ii in data["t"]]

    # -------------------------------------
    # (a) Population
    # -------------------------------------
    axs[0, 0].plot(data["t"],data["yA"],color='k',label=r'i=N$_2$O$_4$')
    axs[0, 0].axhline(y=equilibrium[0] ,color="k",ls=":",zorder=1)

    axs[0, 0].plot(data["t"],data["yB"],color='r',label=r'i=NO$_2$')
    axs[0, 0].axhline(y=equilibrium[1] ,color="r",ls=":",zorder=1)

    axs[0, 0].yaxis.set_major_locator(ticker.MultipleLocator(0.2))
    axs[0, 0].set_ylabel(r'$y_i$',fontsize=14)
    axs[0, 0].set_xlabel(rf'Time ({unitst:s})',fontsize=14)
    axs[0, 0].set_title('(a)')
    axs[0, 0].legend(frameon=False)

    # -------------------------------------
    # (b) ξ vs time
    # -------------------------------------
    if False:
       lim1,lim2 = sorted([data["xi"][0],data["xi"][-1]])
       Dxi       = 0.1*(lim2-lim1)
       xlim1 = lim1-Dxi
       xlim2 = lim2+Dxi
    else:
       xlim1 = xi_min
       xlim2 = xi_max
    axs[0, 1].plot(data["t"],data["xi"],ls='-',color='k')
    axs[0, 1].axhline(y=equilibrium[3] ,ls=":",color="k",zorder=1)

    axs[0, 1].set_ylabel('$\\xi$ (mol)',fontsize=14)
    axs[0, 1].set_ylim(xlim1,xlim2)
    axs[0, 1].yaxis.set_major_formatter(ticker.FormatStrFormatter('%.2f'))

    axs[0, 1].set_xlabel(rf'Time ({unitst:s})',fontsize=14)
    axs[0, 1].set_title('(b)')

    # -------------------------------------
    # (c) Q vs time
    # -------------------------------------
    xx,yy = data["t"][1:], data["Qp"][1:]
    axs[1, 0].plot(xx,yy,ls='-',color='k',zorder=2)
    axs[1, 0].axhline(y=equilibrium[2],ls=":" ,color="k",zorder=1)
    axs[1, 0].set_ylabel('$Q_p^\\circ$',fontsize=14)
    axs[1, 0].set_xlabel(rf'Time ({unitst:s})',fontsize=14)
    axs[1, 0].set_title('(c)')

    # -------------------------------------
    # (d) G/RT vs time
    # -------------------------------------
    if   "G" in data: key = "G"
    elif "A" in data: key = "A"
    else            : key = None
    if key is not None:
       yy = [yi/(R*T) for yi in data[key]]
       axs[1, 1].plot(data["t"],yy,color='k')
       axs[1, 1].axhline(y=equilibrium[4]/(R*T),ls=":",color="k",zorder=1)
       axs[1, 1].set_ylabel(rf'$\left({key:s}(\xi)-{key:s}(0)\right) / (RT)$ (mol)',fontsize=14)
       axs[1, 1].set_xlabel(rf'Time ({unitst:s})',fontsize=14)
       axs[1, 1].set_title('(d)')

    # --- update global variable: last_fig ---
    plt.tight_layout()
    global last_fig
    last_fig = fig

    # --- Show and close figure ---
    fig.set_size_inches(9.0,6.6)
    plt.show()
    plt.close(fig)
#=====================================================

In [ ]:
#@title <small> 💻 Load Helper Functions for the Constant-T,p Scenario<small> { display-mode: "form" }

#=====================================================================#
def Gibbs_xi(xi,T,p,nA0,nB0):
    #Delta_r{G}^o(T)
    DGo_T   = get_constants(T)[0]
    #Delta_r{G}^*(T)
    DGast   = DGo_T + R*T*np.log(p/P_o)
    # mix term
    n0      = nA0 + nB0
    nA      = nA0 -   xi
    nB      = nB0 + 2*xi
    n       = nA  + nB
    DDGmix  = 0.0
    if nA  != 0.0: DDGmix += nA *np.log(nA /n )
    if nB  != 0.0: DDGmix += nB *np.log(nB /n )
    if nA0 != 0.0: DDGmix -= nA0*np.log(nA0/n0)
    if nB0 != 0.0: DDGmix -= nB0*np.log(nB0/n0)
    DDGmix *= R*T
    # return G(xi) - G(0)
    DGtot   = DGast * xi + DDGmix
    return DGtot
#---------------------------------------------------------------------#
def consTP_data_from_xi(xi,T,p,nA0,nB0):
    nA = nA0 - 1*xi
    nB = nB0 + 2*xi
    # check values
    if abs(nA) < ZERO: nA = 0.0
    if abs(nB) < ZERO: nB = 0.0
    n  = nA  + nB
    yA = nA/n
    yB = nB/n

    pA = yA*p
    pB = yB*p

    V  = n * R * T / p
    cA = nA/V
    cB = nB/V

    DGtot = Gibbs_xi(xi,T,p,nA0,nB0)
    return DGtot,n,V,(nA,pA,cA,yA),(nB,pB,cB,yB)
#---------------------------------------------------------------------#
def consTP_dist_from_eq(xi,T,p,nA0,nB0):
    DGtot,n,V,(nA,pA,cA,yA),(nB,pB,cB,yB) = consTP_data_from_xi(xi,T,p,nA0,nB0)
    if pA == 0.0: Qp_o = float("inf")
    else        : Qp_o = (pB*pB)/(pA*P_o)
    # Get distance from equilibrium constant
    Kp_o = get_constants(T)[1]
    return (Kp_o-Qp_o)/Kp_o
#---------------------------------------------------------------------#
def consTP_equilibrium(T,p,nA0,nB0):

    xi_min,xi_max = limits_xi([nA0,nB0],[-1,2])
    xi_guess      = (xi_min+xi_max)/2
    args          = (T,p,nA0,nB0)
    result        = root_scalar(consTP_dist_from_eq,x0=xi_guess,args=args,bracket=(xi_min,xi_max))
    xi_eq         = float(result.root)

    # data at equilibrium
    DGtot,n,V,(nA,pA,cA,yA),(nB,pB,cB,yB) = consTP_data_from_xi(xi_eq,T,p,nA0,nB0)
    Kp_o = get_constants(T)[1]

    # checking: Qp should be equal to Kp_o (0.1% of error accepted)
    Qp_eq = pB**2 / (pA*P_o)
    rdiff = 100*abs(Kp_o-Qp_eq)/Kp_o
    assert rdiff < 0.1

    # return data
    return xi_eq,n,V,DGtot,(nA,pA,cA,yA),(nB,pB,cB,yB)
#---------------------------------------------------------------------#
def consTP_at_given_time(t,A,x0,x1,x2,T,p,nA0,nB0):
    '''
    Relationship between yB (from differential eq) and xi:
        nB=nB0+2xi            --> yB=(nB0+2xi)/n       -->
    --> yB=(nB0+2xi)/(n0+xi)  --> yB*n0+yB*xi=nB0+2xi --->
    --> (yB-2)*xi=(nB0-yB*n0) --> xi = (nB0-yB*n0)/(yB-2)
    '''

    # mole fractions
    yB = solution_diff_eq(t,A,x0,x1,x2)
    if abs(yB)     < ZERO: yB = 0.0
    if abs(yB-1.0) < ZERO: yB = 1.0

    # Get extent of reaction
    n0  = nA0+nB0
    xi  = (nB0-yB*n0)/(yB-2)

    # Number of moles
    DGtot,n,V,(nA,pA,cA,yA),(nB,pB,cB,yB) = consTP_data_from_xi(xi,T,p,nA0,nB0)

    # Reaction quotients
    Qp = pB**2 / (pA*P_o) if pA != 0. else float("inf")
    Qc = cB**2/  (cA*c_o) if cA != 0. else float("inf")

    # Return data
    return DGtot, (xi,n,V), (nA,pA,cA,yA),(nB,pB,cB,yB), (Qp,Qc)
#---------------------------------------------------------------------#
def consTP(T,p,nA0,nB0):

    # Get equilibrium constants and rate constants
    DGo,Kp_o,Kc_o,kfw,kbw,string_constants = get_constants(T)
    STRING = string_constants

    # Other constants of interest
    Kc = kfw/kbw

    # Obtain constants for diff equation: A, x1, x2 and x0
    A  = 2*kbw*p/(R*T)
    x0 = nB0/(nA0+nB0)
    x1 = -Kc*R*T + np.sqrt(Kc*R*T*(Kc*R*T+4*p)); x1 /= (2*p)
    x2 = -Kc*R*T - np.sqrt(Kc*R*T*(Kc*R*T+4*p)); x2 /= (2*p)

    # initial conditions
    args0       = (0,A,x0,x1,x2,T,p,nA0,nB0)
    data0       = consTP_at_given_time(*args0)
    DGtot0      = data0[0]
    xi0,n0,V0   = data0[1]
    pA0,cA0,yA0 = data0[2][1:4]
    pB0,cB0,yB0 = data0[3][1:4]
    Qp0,Qc0     = data0[4]
    xi0         = 0.0 # to avoid negative value (-0.0)
    args = (T,p,V0,xi0,n0,nA0,yA0,pA0,cA0,nB0,yB0,pB0,cB0,Kp_o)
    STRING += rf"   * Initial conditions:"+"\n\n"
    STRING += string_conditions(*args,DG=DGtot0)+"\n"

    # values at equilibrium
    data_eq = consTP_equilibrium(T,p,nA0,nB0)
    xi_eq,n_eq ,V_eq ,DG_eq = data_eq[0:4]
    nA_eq,pA_eq,cA_eq,yA_eq = data_eq[4]
    nB_eq,pB_eq,cB_eq,yB_eq = data_eq[5]

    args = (T,p,V_eq,xi_eq,n_eq,nA_eq,yA_eq,pA_eq,cA_eq,nB_eq,yB_eq,pB_eq,cB_eq,Kp_o)
    STRING += rf"   * At equilibrium:"+"\n\n"
    STRING += string_conditions(*args,DG=DG_eq)+"\n"

    # how much for 99% of equilibrium?
    if xi_eq == 0.0:
       tmax = 1E-3
    else:
       xi   = xi_eq * REL_XI_EQ
       nA   = nA0 -   xi
       nB   = nB0 + 2*xi
       yB   = nB/(nA+nB)
       arg  = abs((yB-x1)*(x0-x2)/(yB-x2)/(x0-x1))
       teq  = - np.log(arg)/A/(x1-x2)
       tmax = 2*teq

       args = (A,x0,x1,x2,T,p,nA0,nB0)
       xi   = consTP_at_given_time(tmax,*args)[1][0]

    # Solve equation at each time
    saved = {"t":np.linspace(0,tmax,NPOINTS)}
    for t in saved["t"]:
        args = (A,x0,x1,x2,T,p,nA0,nB0)
        data = consTP_at_given_time(t,*args)

        DG          = data[0]
        xi,n,V      = data[1]
        nA,pA,cA,yA = data[2]
        nB,pB,cB,yB = data[3]
        Qp,Qc       = data[4]

        # Save data
        saved["Qp"] = saved.get("Qp",[]) + [Qp]
        saved["xi"] = saved.get("xi",[]) + [xi]
        saved["yA"] = saved.get("yA",[]) + [yA]
        saved["yB"] = saved.get("yB",[]) + [yB]
        saved["G" ] = saved.get("G" ,[]) + [DG]

    # Plot data of interest
    equilibrium   = (yA_eq,yB_eq,Kp_o,xi_eq,DG_eq)
    xi_min,xi_max = limits_xi([nA0,nB0],[-1,2])
    plot_data(T,saved,equilibrium,xi_min,xi_max)

    return STRING
#=====================================================================#

In [ ]:
#@title <small> 💻 Load Helper Functions for the Constant-T,V Scenario<small> { display-mode: "form" }
#=====================================================================#
def Helmholtz_xi(xi,T,V,nA0,nB0):
    # initial conditions
    n0      = nA0 + nB0
    p0      = n0*R*T/V
    # current conditions
    nA      = nA0 -   xi
    nB      = nB0 + 2*xi
    n       = nA  + nB
    p       = n *R*T/V
    #Delta_r{G}^o(T)
    DGo_T   = get_constants(T)[0]
    # pressure term
    termP   = p *np.log(p /P_o/np.e)
    termP  -= p0*np.log(p0/P_o/np.e)
    # mix term
    DDGmix  = 0.0
    if nA < 0 or nB < 0: print(nA,nB)
    if nA  != 0.0: DDGmix += nA *np.log(nA /n )
    if nB  != 0.0: DDGmix += nB *np.log(nB /n )
    if nA0 != 0.0: DDGmix -= nA0*np.log(nA0/n0)
    if nB0 != 0.0: DDGmix -= nB0*np.log(nB0/n0)
    DDGmix *= R*T
    # return A(xi) - A(0)
    DAtot   = DGo_T * xi + V*termP + DDGmix
    return DAtot
#---------------------------------------------------------------------#
def consTV_data_from_xi(xi,T,V,nA0,nB0):
    nA = nA0 - 1*xi
    nB = nB0 + 2*xi
    # check values
    if abs(nA) < ZERO: nA = 0.0
    if abs(nB) < ZERO: nB = 0.0
    n  = nA  + nB
    p  = n * R * T / V
    yA = nA/n
    pA = yA*p
    cA = nA/V
    yB = nB/n
    pB = yB*p
    cB = nB/V
    DAtot = Helmholtz_xi(xi,T,V,nA0,nB0)
    return DAtot,n,p,(nA,pA,cA,yA),(nB,pB,cB,yB)
#---------------------------------------------------------------------#
def consTV_dist_from_eq(xi,T,V,nA0,nB0):
    DAtot,n,p,(nA,pA,cA,yA),(nB,pB,cB,yB) = consTV_data_from_xi(xi,T,V,nA0,nB0)
    if pA == 0.0: Qp_o = float("inf")
    else        : Qp_o = (pB*pB)/(pA*P_o)
    # Get distance from equilibrium constant
    Kp_o = get_constants(T)[1]
    return (Kp_o-Qp_o)/Kp_o
#---------------------------------------------------------------------#
def consTV_equilibrium(T,V,nA0,nB0):

    xi_min,xi_max = limits_xi([nA0,nB0],[-1,2])
    xi_guess      = (xi_min+xi_max)/2
    args          = (T,V,nA0,nB0)
    result        = root_scalar(consTV_dist_from_eq,x0=xi_guess,args=args,bracket=(xi_min,xi_max))
    xi_eq         = float(result.root)

    # data at equilibrium
    DAtot,n,p,(nA,pA,cA,yA),(nB,pB,cB,yB) = consTV_data_from_xi(xi_eq,T,V,nA0,nB0)
    Kp_o = get_constants(T)[1]

    # checking: Qp should be equal to Kp_o (0.1% of error accepted)
    Qp_eq = pB**2 / (pA*P_o)
    rdiff = 100*abs(Kp_o-Qp_eq)/Kp_o
    assert rdiff < 0.1

    # return data
    return xi_eq,n,p,DAtot,(nA,pA,cA,yA),(nB,pB,cB,yB)
#---------------------------------------------------------------------#
def consTV_at_given_time(t,A,x0,x1,x2,T,V,nA0,nB0):

    # number of mole for B
    cB = solution_diff_eq(t,A,x0,x1,x2)
    if abs(cB) < ZERO: cB = 0.0
    nB = cB*V

    # Get extent of reaction
    xi = (nB-nB0)/2

    # data at the given extent
    DAtot,n,p,(nA,pA,cA,yA),(nB,pB,cB,yB) = consTV_data_from_xi(xi,T,V,nA0,nB0)

    # Reaction quotients
    Qp = pB**2 / (pA*P_o) if pA != 0 else float("inf")
    Qc = cB**2/  (cA*c_o) if cA != 0 else float("inf")

    # Return data
    return DAtot, (xi,n,p), (nA,pA,cA,yA),(nB,pB,cB,yB), (Qp,Qc)
#---------------------------------------------------------------------#
def consTV(T,V,nA0,nB0):

    STRING = ""

    # Get equilibrium constants and rate constants
    DGo,Kp_o,Kc_o,kfw,kbw,string_constants = get_constants(T)
    STRING += string_constants

    # Other constants of interest
    Kc   = kfw/kbw

    cA0,cB0 = nA0/V, nB0/V
    # Obtain constants for diff equation: A, x1, x2 and x0
    A  = 2*kbw
    x0 = nB0/V
    x1 = -Kc + np.sqrt(Kc*(Kc+8*(2*cA0+cB0))); x1 /= 4
    x2 = -Kc - np.sqrt(Kc*(Kc+8*(2*cA0+cB0))); x2 /= 4

    # initial conditions
    args0       = (0,A,x0,x1,x2,T,V,nA0,nB0)
    data0       = consTV_at_given_time(*args0)
    DAtot0      = data0[0]
    xi0,n0,p0   = data0[1]
    pA0,cA0,yA0 = data0[2][1:4]
    pB0,cB0,yB0 = data0[3][1:4]
    Qp0,Qc0     = data0[4]
    xi0         = 0.0 # to avoid negative value (-0.0)

    args = (T,p0,V,xi0,n0,nA0,yA0,pA0,cA0,nB0,yB0,pB0,cB0,Kp_o)
    STRING += rf"   * Initial conditions:"+"\n\n"
    STRING += string_conditions(*args,DA=DAtot0)+"\n"

    # values at equilibrium
    data_eq = consTV_equilibrium(T,V,nA0,nB0)
    xi_eq,n_eq ,p_eq ,DA_eq = data_eq[0:4]
    nA_eq,pA_eq,cA_eq,yA_eq = data_eq[4]
    nB_eq,pB_eq,cB_eq,yB_eq = data_eq[5]

    args = (T,p_eq,V,xi_eq,n_eq,nA_eq,yA_eq,pA_eq,cA_eq,nB_eq,yB_eq,pB_eq,cB_eq,Kp_o)
    STRING += rf"   * At equilibrium:"+"\n\n"
    STRING += string_conditions(*args,DA=DA_eq)+"\n"

    # how much for 99% of equilibrium?
    if xi_eq == 0.0:
       tmax = 1E-3
    else:
       xi   = xi_eq * REL_XI_EQ
       nB   = nB0 + 2*xi
       cB   = nB/V
       arg  = abs((cB-x1)/(cB-x2) * (x0-x2)/(x0-x1))
       teq  = - np.log(arg)/A/(x1-x2)
       tmax = 2*teq

    # Solve equation at each time
    saved = {"t":np.linspace(0,tmax,NPOINTS)}
    for t in saved["t"]:
        args = (A,x0,x1,x2,T,V,nA0,nB0)
        data = consTV_at_given_time(t,*args)

        DA          = data[0]
        xi,n,p      = data[1]
        nA,pA,cA,yA = data[2]
        nB,pB,cB,yB = data[3]
        Qp,Qc       = data[4]

        # Save data
        saved["Qp"] = saved.get("Qp",[]) + [Qp]
        saved["xi"] = saved.get("xi",[]) + [xi]
        saved["yA"] = saved.get("yA",[]) + [yA]
        saved["yB"] = saved.get("yB",[]) + [yB]
        saved["A" ] = saved.get("A" ,[]) + [DA]

    # Plot data of interest
    equilibrium   = (yA_eq,yB_eq,Kp_o,xi_eq,DA_eq)
    xi_min,xi_max = limits_xi([nA0,nB0],[-1,2])
    plot_data(T,saved,equilibrium,xi_min,xi_max)

    return STRING
#=====================================================================#

### **The paradigmatic example**: the $\rm{N_2O_4(g)}\ \rightleftharpoons\ 2 NO_2(g)$ reaction



####
Let us revisit, for the final time, the dissociation reaction of $\rm N_2O_4 (g)$:

$$\rm N_2O_4 (g) \rightleftharpoons 2 \; NO_2 (g)$$

In this Notebook, we will analyze the connection between **chemical equilibrium** and **chemical kinetics**.

Our goal is to explore how equilibrium is reached as a _dynamic process_. We will consider the two classical scenarios: constant-_T,p_ and constant-_T,V_.

####
**(a) Thermodynamic background**

To describe how the equilibrium constant varies with temperature, we will use the same expressions introduced in _Notebook 1_. The temperature dependence of the standard Gibbs free energy change is given by:

$$
\Delta_{r} G^{\circ}(T)= \Delta_{r} H^{\circ}(T_{\rm ref})-T \cdot \Delta_{r} S^{\circ}(T_{\rm ref})+\Delta_{r} C_p^\circ \cdot \left[ T-T_{\rm ref}+T \cdot \ln \left(\frac{T_{\rm ref}}{T}\right)\right]
 \tag{1}
$$

assuming $\Delta_{r} C_p^\circ$ is constant over the temperature range considered.
From this, the equilibrium constant follows directly as:

$$
K_p^\circ(T) = e^{-\Delta_{r} G^{\circ}(T)/(R \cdot T)}
 \tag{2}
$$

We will use the following experimental thermodynamic data at the reference temperature ($T_{\rm ref}=298$ K):
- $\Delta_{r} H^{\circ}(T_{\rm ref}) = \;\;57.20$ kJ mol$^{-1}$,
- $\Delta_{r} S^{\circ}(T_{\rm ref}) \;= 175.83$ J mol$^{-1}$ K$^{-1}$ and
- $\Delta_{r} C_p^\circ(T_{\rm ref}) \;= -2.88$ J mol$^{-1}$ K$^{-1}$.

In [ ]:
#@title <small> <small> { display-mode: "form" }
T_ref    = 298
DHo_ref  =  57.20E3
DSo_ref  = 175.83
DCPo_ref =  -2.88

####
**(b) Kinetic background**

To connect kinetics with thermodynamics, we require the forward ($k_{fw}$) and backward ($k_{bw}$) rate constants. For the reaction here studied, these rate constants are related to the equilibrium constant according to:

$$
K_p^\circ(T) \cdot \frac{p^\circ}{RT} = \frac{k_{fw}}{k_{bw}}
 \tag{3}
$$

where $p^\circ = 1$ bar is the standard pressure. Thus, if the temperature dependence of $k_{fw}(T)$ is known, the backward rate constant follows immediately.

For the forward reaction, we can adopt an Arrhenius-type expression:

$$
k_{fw} = A \cdot e^{-B/T}
 \tag{4}
$$

where $A$ is the pre-exponential factor and $B$ is related to the activation energy. In particular, we will use the parametrization reported in _Atmos. Chem. Phys._ **4**, 1461-1738 (2004)[[🌐]](https://doi.org/10.5194/acp-4-1461-2004):

$$
k_{fw} = 1.15 \cdot 10^{16}  \cdot e^{-6460/T} \;\;\; s^{-1}
 \tag{5}
$$

Although this expression is formally valid only in the 250–300 K range and corresponds to the high-pressure limit, we will apply it over the full temperature interval for pedagogical purposes.

<br>

_You may adjust any parameter in the next cell prior to execution if you wish to explore alternative data or hypothetical scenarios._

In [ ]:
#@title <small> <small> { display-mode: "form" }
arrhenius_A = 1.15E16 # in 1/s
arrhenius_B = 6460    # in K

#### **(c) Carrying out the experiments**

Use the following cell to configure and analyze a scenario by selecting the initial temperature, pressure, and volume of the system. You may also specify the initial mole fraction of the reactant (N$_2$O$_4$) and choose whether the experiment is performed under constant-_T,p_ or constant-_T,V_ conditions.
The plots will update dynamically as you adjust the initial conditions using the sliders.

In [ ]:
#@title <small> <small> { display-mode: "form" }

#=====================================================
def simulate(T0,p0,V0,yA0,scenario):
    # --- global variable to update ---
    global last_info
    # initial number of moles
    yB0 = 1.0-yA0
    n0  = p0*V0/(R*T0)
    nA0 = n0*yA0
    nB0 = n0*yB0
    if   scenario == "TP": last_info = consTP(T0,p0,nA0,nB0)
    elif scenario == "TV": last_info = consTV(T0,V0,nA0,nB0)
    else                 : raise Exception
#=====================================================

# Enable Colab’s custom widget manager, allowing interactive ipywidgets to function correctly
output.enable_custom_widget_manager()

# -------- Sliders --------
args      = dict(layout=w.Layout(width='600px'),style={'description_width': '150px'},continuous_update=True,readout_format='.2f')
T_slider  = w.FloatSlider(value=298.00,min=200.00,max=400.00,step=1.00,description=r'T [K]'  , **args)
P_slider  = w.FloatSlider(value=  1.00,min=  0.01,max=  3.00,step=0.01,description=r'p [bar]', **args)
V_slider  = w.FloatSlider(value= 24.78,min=  2.00,max=250.00,step=0.01,description=r'V [L]', **args)
yA_slider = w.FloatSlider(value=  1.00,min=  0.00,max=  1.00,step=0.01,description=r'y(N2O4)', **args)

# -------- Scenario selector --------
selector = w.RadioButtons(
    options=[('constant-T,p', 'TP'),
             ('constant-T,V', 'TV')],
    style={'description_width': '100px'},
    layout=w.Layout(width='260px', margin='0 0 0 40px')
)

# -------- download button --------
btn = w.Button(description='Download current figure', icon='download', button_style='primary',layout=w.Layout(width='200px', height='30px'))
btn.on_click(lambda b: _on_download_clicked(b,selector.value))

# -------- slider ---> function --------
# P*1E+5 : bar --> Pa
# V*1E-3 : L   --> m3
string = w.Output()
out    = w.interactive_output(lambda T0, P0, V0, yA0, scenario: simulate(T0, P0*1E5, V0*1E-3,yA0,scenario), {'T0': T_slider, 'P0': P_slider,'V0': V_slider,'yA0': yA_slider,'scenario':selector})

# Print sliders and selector
print("Set the initial conditions:")
ui = w.VBox([T_slider,P_slider,V_slider,yA_slider])
display(ui)
print("Select scenario:")
display(w.VBox([selector]))

# Print download button and plot
display(w.VBox([btn,out]))





#####

By running the next cell, the information about both the initial state and the equilibrium state corresponding to the most recent experiment configured in the previous cell will be displayed.

In [ ]:
#@title <small> <small> { display-mode: "form" }
print(last_info)